In [ ]:
!pip install torch torchvision
!pip install matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
root_dir = '/content/drive/MyDrive/Colab Notebooks/LLVIP'

Mounted at /content/drive


In [ ]:
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, Lambda
import torchvision.transforms.functional as TF

# 定义可见光图像的转换
visible_transforms = Compose([
    Resize((800, 800)),  # Resize images if required
    ToTensor(),  # Convert images to PyTorch tensors
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize for pretrained models
])

# 定义红外图像的转换
infrared_transforms = Compose([
    Resize((800, 800)),  # Resize images if required
    ToTensor(),  # Convert images to PyTorch tensors
    Normalize(mean=[0.5], std=[0.5])  # Normalize for single-channel (grayscale)
])


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import torch
from torchvision.transforms import Compose, ToTensor, Resize
from PIL import Image
import xml.etree.ElementTree as ET

class LLVIPDataset(torch.utils.data.Dataset):
    def __init__(self, root, subset="train", visible_transforms=None, infrared_transforms=None):
        self.root = root
        self.subset = subset
        self.visible_transforms = visible_transforms
        self.infrared_transforms = infrared_transforms
        self.infrared_images = list(sorted(os.listdir(os.path.join(root, "infrared", subset))))
        self.visible_images = list(sorted(os.listdir(os.path.join(root, "visible", subset))))
        self.annotations = list(sorted(os.listdir(os.path.join(root, "Annotations", subset))))

    def __getitem__(self, idx):
        infrared_path = os.path.join(self.root, "infrared", self.subset, self.infrared_images[idx])
        visible_path = os.path.join(self.root, "visible", self.subset, self.visible_images[idx])
        annotation_path = os.path.join(self.root, "Annotations", self.subset, self.annotations[idx])

        # Load images
        infrared_img = Image.open(infrared_path).convert("L")
        visible_img = Image.open(visible_path).convert("RGB")

        # Apply transformations
        if self.infrared_transforms:
            infrared_img = self.infrared_transforms(infrared_img)
        if self.visible_transforms:
            visible_img = self.visible_transforms(visible_img)

        #print(f"Image types after conversion - Visible: {type(visible_img)}, Infrared: {type(infrared_img)}")

        # Parse annotations
        boxes = []
        tree = ET.parse(annotation_path)
        root = tree.getroot()
        for obj in root.findall('object'):
            bbox = obj.find('bndbox')
            xmin = float(bbox.find('xmin').text)
            ymin = float(bbox.find('ymin').text)
            xmax = float(bbox.find('xmax').text)
            ymax = float(bbox.find('ymax').text)
            if xmin < xmax and ymin < ymax:
              boxes.append([xmin, ymin, xmax, ymax])
        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.ones((len(boxes),), dtype=torch.int64)

        return visible_img, infrared_img, boxes, labels

    def __len__(self):
        return len(self.visible_images)


In [ ]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision

def get_model(num_classes):
    # 加载预训练的Faster R-CNN模型
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

    # 替换头部以适应额外的通道
    in_channels = model.backbone.out_channels
    model.backbone.out_channels = 4
    model.roi_heads.box_predictor = FastRCNNPredictor(in_channels, num_classes)

    return model

# 实例化模型和其他组件
model = get_model(num_classes=2)  # 1 类 + 背景


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:01<00:00, 151MB/s]


In [ ]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.transforms import functional as F

# 使用梯度裁剪
def train_one_epoch(model, optimizer, data_loader, device):
    model.train()
    total_loss = 0
    for images, targets in data_loader:
        images = list(img.to(device) for img in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        if torch.isnan(losses).any():
            print("NaN detected in losses")

        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 梯度裁剪
        optimizer.step()

        total_loss += losses.item()
    return total_loss / len(data_loader)

def evaluate(model, data_loader, device):
    model.eval()  # 确保模型处于评估模式
    total_loss = 0

    with torch.no_grad():
        for images, targets in data_loader:
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # 临时切换到训练模式以计算损失
            model.train()
            loss_dict = model(images, targets)
            model.eval()  # 切回评估模式

            losses = sum(loss for loss in loss_dict.values())
            total_loss += losses.item()

    return total_loss / len(data_loader)


In [ ]:
# 设定设备
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
if torch.cuda.is_available():
    print("CUDA is available. Using GPU for training.")
else:
    print("CUDA is not available. Using CPU for training.")


# 获取模型
model = fasterrcnn_resnet50_fpn(pretrained=True)
num_classes = 2  # 1 类（行人）+ 1 类背景
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# 将模型移到设备上
model.to(device)

# 定义优化器
optimizer = optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)



CUDA is available. Using GPU for training.


In [ ]:
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

# 定义图像转换
transformations = Compose([
    Resize((800, 800)),  # Resize images if required
    ToTensor(),  # Convert images to PyTorch tensors
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize for pretrained models
])

dataset_train = LLVIPDataset(root='/content/drive/MyDrive/LLVIP', subset='train',
                             visible_transforms=visible_transforms, infrared_transforms=infrared_transforms)
dataset_val = LLVIPDataset(root='/content/drive/MyDrive/LLVIP', subset='test',
                           visible_transforms=visible_transforms, infrared_transforms=infrared_transforms)






In [ ]:
from torch.utils.data import DataLoader
from torchvision.transforms.functional import to_tensor

def fuse_images(rgb_image, ir_image, alpha=0.3):
    """
    Fuse an infrared image into the RGB image.
    :param rgb_image: A tensor of the RGB image.
    :param ir_image: A tensor of the IR image (single channel).
    :param alpha: Weight factor for the infrared contribution.
    :return: A tensor of the fused RGB image.
    """
    #print(f"Original RGB shape: {rgb_image.shape}")  # 应为 [3, H, W]
    #print(f"Original IR shape: {ir_image.shape}")    # 应为 [1, H, W]

    # 正确扩展IR图像
    ir_image_expanded = ir_image.repeat(3, 1, 1)  # Repeat函数复制单通道以匹配RGB的三通道

    # 执行加权融合
    fused_image = alpha * ir_image_expanded + (1 - alpha) * rgb_image
    return fused_image

def my_collate_fn(batch):
    images, targets = [], []
    for item in batch:
        visible_img, infrared_img, boxes, labels = item

        # 直接融合已经是张量的图像
        fused_img = fuse_images(visible_img, infrared_img, alpha=0.3)

        # 添加融合后的图像和对应的目标到列表
        images.append(fused_img)
        targets.append({'boxes': boxes, 'labels': labels})

    # 将图像列表堆叠成一个批次
    images = torch.stack(images, dim=0)

    return images, targets


"""
def my_collate_fn(batch):
    visible_images = [item[0] for item in batch]
    infrared_images = [item[1] for item in batch]
    boxes = [item[2] for item in batch]
    labels = [item[3] for item in batch]

    # 此处可以添加图像融合的代码或者其他处理
    # 现在假设我们只处理可见光图像
    images = torch.stack(visible_images, dim=0)

    # 对targets进行合适的处理
    targets = []
    for b, l in zip(boxes, labels):
        targets.append({'boxes': b, 'labels': l})

    return images, targets
"""


# 假设 dataset_train 和 dataset_val 是已经初始化的数据集实例
data_loader_train = DataLoader(dataset_train, batch_size=10, shuffle=True, num_workers=2, collate_fn=my_collate_fn)
data_loader_val = DataLoader(dataset_val, batch_size=10, shuffle=False, num_workers=2, collate_fn=my_collate_fn)







In [ ]:
num_epochs = 10  # 可以根据需要调整这个数字
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
for epoch in range(num_epochs):
    # 训练一个epoch
    train_loss = train_one_epoch(model, optimizer, data_loader_train, device)
    print(f"Epoch {epoch} Train loss: {train_loss:.4f}")

    # 验证模型性能
    val_loss = evaluate(model, data_loader_val, device)
    print(f"Epoch {epoch} Validation loss: {val_loss:.4f}")


Epoch 0 Train loss: 2.7272
Epoch 0 Validation loss: 3.6684
Epoch 1 Train loss: 2.6138
Epoch 1 Validation loss: 3.6418
Epoch 2 Train loss: 2.5667
Epoch 2 Validation loss: 3.5465
Epoch 3 Train loss: 2.5521
Epoch 3 Validation loss: 3.5542
Epoch 4 Train loss: 2.5179
Epoch 4 Validation loss: 3.5850
Epoch 5 Train loss: 2.4925
Epoch 5 Validation loss: 3.5165
Epoch 6 Train loss: 2.4915
Epoch 6 Validation loss: 3.5024
